## Data Version Control (DVC)
### What is Data Versioning?
* Definition
  + monitors data changes over time
  + snapshots data over iterations
  + similar to coder versioning
* Benifits
  + retrieval and scrutiny
  + data cosistency, accountability, and lineage
* Applications
  + data science and machining learning
  + data engineering
  + financial analysis, auditing and compliance
* Example in machine learning:
  + both changes in hyperparameters and dataset change the model performance metrics

#### Git as version control
* code version control system
* independent local development
  + branch and merge
  + version history management
* enables collaboration
* run on terminal and tracks
  + actual files/folders to be tracked
  + git metadata (in .git folder)

#### Data Version Control (DVC)
* manages data and experiments
* similar to git
* git tracks metadata, DVC handles data versioning

#### Git vs. DVC CLI
```bash
# initialize repos in working folder
# Both of them point to the same codebase
# git focuses on code, dvc on datasets
git init # initialize code repo
dvc init # initialize data repo. 

# adding files/data to repos
git add code.py
dvc add data/mydata.csv

# commit changes
git commit -m "adding first file"
dvc commit 

# push code/data to remote server
git push
dvc push

# pulling changes from remote, or synchronizing dvc project
git pull
dvc pull

# cloing an existing repo from remote, or download a file or directory tracked by dvc
git clone https://github.com/username/repo-name.git
dvc get https://github.com/username/repo-name model.pkl

# update all DVC-tracked files and directories to match a specific state
# example: when you change branches using git checkout, git updates .dvc files
# but your workspace data files don't automatically update. So your workspace
# becomes out of sync. dvc checkout fiexed that
git checkout another_branch
dvc checkout
  

### What DVC can do in machine learning pipeline?
* capture data and models in git by versioning metadata files that only keep track of changes
* using DVC pipelines to define and execute ML workflows
  + step by step guide for a machine learning or data task. It defines for each step:
    + what needs to be done
    + in what order
    + what we need as dependency
    + output at the end of the step
  + pipelines are defined as key value pairs in yaml files such as dvc.yaml
  + an example of dvc.yaml section
```YAML
stages:
  train:                           # step name
    cmd: python train.py           # command to run this step
    deps:                          # dependencies required (code, data and params)
      - code/train.py
      - data/input_data.csv
      - params/params.json
    outs:                          # output (trained model)
      - model_output/model.pkl
```
  + you run pipelines by `dvc repro`
* we can use dvc to track metrics and plots using `dvc metrics diff`
  + once there is any changes in metrics and plots, dvc will track them
  + for example, it can list the difference between head and current workspace of tracked AUC metric
  + we can run `dvc exp save` after `dvc repro` to log metrics. we can use `dvc exp run` to combine these two steps
  + we can visualize these experiments by `dvc exp show` in a table
  + experiments are custom git references that prevent bloating up git commits
* we use DVC as data registry
  + build a centralized data store that serve multiple projects
  + with the help of git to track version metadata and back actual data in data storage  

### Practice
#### Installation
* install by `pip install dvc`
  + in a virtual environment
  + ensure git is installed
* verify installation
  `dvc version`
* ensure git is initialized in the current ropo, otherwise run `git init`
* initialize dvc by `dvc init`
* initialization creates initial files that should be tracked by git. you can see new added files by
  `git status`
* then you commit the changes by `git commit -m "initialized dvc"`
* you can define a .dvcignore file that outlines files/directories that DVC will ignore.
  ```bash
  # example of .dvcignore
  # ignore all files in 'data' directory
  data/*

  # but don't ignore 'data/data.csv'
  !data/data.csv

  # ignore all .tmp file
  *.tmp
  ```
  + you can check if a specific file is excluded by `dvc check-ignore <filename>`

#### DVC cache
* helps manage and version large dataset and models
* a hidden storage for files and folders tracked by dvc and their different versions
* it stores temporary files until commited, allowing us to work with large datasets without committing them until they are ready
* no restrictions of what files can be versioned with DVC but usually we use it for large datasets and non-text (binary) files
* it usually resides under the .dvc directory in the workspace of a dvc project but is configurable with `dvc cache dir` command. For example `dvc cache dir ~/mycache`

* what happens when adding a new file to dvc cache
  + run `dvc add data.csv`
  + dvc creates a new text-readable file data.csv.dvc inside .dvc folder of workspace, which saves metadata about the newly added file
  + moves data to the cache and links it back to our workspace
  + by versioning these .dvc files with Git, we can keep track of data changes while keep the repo lightweight
    + use `git add data.csv.dvc .gitignore`. this will allow git to track changes of data.csv without tracking/saving data.csv file itself
  + .dvc files are unique for each data file tracked by DVC. Each of tracked file has its corresponding .dvc file
  + to version or track the data file, we need to commit the associated .dvc file use `git commit -m "data.csv.dvc"`
* .dvc file content
  + contains a 'outs:' section
    + md5: contains md5 with MD5 checksum of the tracked data file
      + unique value generated based on the contents of the data file
    + size: size of the tracked data file in bytes
    + hash: type of hash function (md5)
    + path: the path to the tracked file within the project directory
* to remove the added file,
  + use dvc remove on the .dvc files. this marks the data file to be removed from cache 
  `dvc remove data.csv.dvc`
    + this removes data.csv.dvc file, but .dvc/cache folder still keeps the link to the .dvc file
  + then use `dvc gc -w` to remove data file from workspace (-w means workspace)
    + this removes the link in .dvc/cache folder such as .dvc/cache/e6/02b...
* to summarize,
  + in local workspace, by using `dvc init` we have
    + project folder as your code repo inside workspace
    + a hidden storage for cache as a temporary staging area with .dvc to track files and directories and their versions
      + /path/to/project/.dvc/cache
  + by `dvc add`, we add content to DVC cache

#### DVC remotes
* to share locally staged artifacts with collaborators, we use centralized external locations as remotes
* once set up, we can move data staged in DVC cache to and from the remotes
* benefits of using remotes
  + synchronize large files and directories
  + centralize or distribute data storage
  + archiving diverse iterations of datasets and models and thereby save local storage capacity

* to set up a DVC remotes
  + add remote by `dvc remote add` command
    ```bash
    # dvc remote add <name> <location>
    dvc remote add -d storage s3://dvc-data # -d means default
    ```
  + this will update remote section of the project's config file .dvc/config
    + default remote is set in the core section of .dvc/config file
  + we can add local remotes using system directories, file systems on laptops, mounted drives, network resources
    ```bash
    # dvc remote add --local <remote_name> </path/to/folder>
    dvc remote add --local local_remote /temp/dvc
    dvc remote modify --local local_remote url /temp/dvc_new
    ```
 
* we can list all DVC remotes by `dvc remote list`
  + it will list all remotes we set up with <remote_name> : <url or folder_path>
* we can modify remote configuration by `dvc remote modify`
  `dvc remote modify s3_remote connection_timeout 300`

* to update data files in staged cache
  ```bash
  dvc push data.csv # for specific data file
  dvc push          # update entire cache

  dvc pull #  downloads tracked data files from remote to cached and links or copies files or folder to workspace
  dvc push -r aws_remote data.csv # override default remote with -r flag.we push to aws_remote, not the default
  ```

* versioning data
  + to retreive a specific version of data, first checkout the corresponding .dvc by git and then data by dvc
    ```bash
    # .dvc is tracked by git, not dvc. First checkout the specific git version
    git checkout <commit_hash|tag|branch>

    # retrieve data with MD5 specified in .dvc file
    dvc checkout <target>
    ```

* procedure to add data in dvc
  ```bash
  # add dataset changes
  dvc add <target>

  # run git status to find out if it is necessary to add .gitignore in the next step
  git status # check if .gitignore is listed as modified, if so add it to git add

  # commit changed .dvc to git
  git add <target>.dvc .gitignore # if .gitignore is listed as modified
  git add <target>.dvc            # if .gitignore is not listed as modified

  # commit the chagnes in git
  git commit <target>.dvc -m "dataset updates"
  git push origin main

  # upload the changed data file to remote
  dvc push
  ```
    

#### DVC pipelines
* Code organization and refactoring
  + prototying code allows rapid iteration
  + Not suitable for production
    + untested and prone to errors
    + not modular with many repeated code blocks
    + likely not reproducible
* Features of good production code
  + reproducible: recreate same outputs in different environments and time
  + modular: written as distinct, independent, and testable modules
  + consistent: single source of truth for all parameters
    + A configuration/parameter file
* configuration files and YAML
  + Files should be in supported format
    + YAML, JSON, TOML, Python
    + Default is params.yaml
  + YMAL file
    + allows a standard format to transfer data between languages or applications
    + simple and clean format
    + valid file extensions: .yaml or .yml
* YAML syntax
  + specify parameters as dictionaries as keys and values separated by :
  + comments starts with #
  + data types:
    + integer, float, string
  + data structure:
    + arrays
    + nested dictionaries
      ```YAML
      a: [1, 2.2, 3, 4.8]
      b:
        - 5
        - "String value"
      c:
        b: "Some value"
        c: "Some other value"
      ```
  + indentation is import
* project folder organization
  + params.yaml: configuration file
  + metrics_and_plots.py: helper functions for model evaluation and plots
  + model.py: model definition
  + preprocess_dataset.py: driver code to preprocess
  + train_and_evaluation.py: driver code to train
  + utils_and_constants.py: utility helper functions
* define params.yaml file
  + the file that contains all parameters across different stages of DVC pipeline
  + an example of params.yaml
    ```YAML
    preprocess:
      # Add Date as a column to drop
      drop_columns:
        - Date
      # Add RainTomorrow as target column
      target_column: RainTomorrow
      categorical_features:
        - Location
        - WindGustDir
        - WindDir9am
        - WindDir3pm
        - RainToday
      train_and_evaluate:
        target_column: RainTomorrow
        train_test_split:
          test_size: 0.2
          random_state: 1993
        shuffle: true
        shuffle_random_state: 1993
        # Add 2 as number of estimators for rfc classifier
        rfc_params:
          n_estimators: 2
          max_depth: 2
          random_state: 42
    ```      

#### Define DVC Pipelines
  + sequence of stages defining machine learning workflow and dependencies
    + Versioned and tracked with Git
  + Defined in dvc.yaml file including the following sections
    + input data and scripts (deps)
    + parameters (params)
      + where we define both params file name and the specific key
      + only chagnes in this key will affect this step to re-execute
      + after we define the parameter file in the first stage, the following stage will automatically assume the same parameter file
    + stage execution commands (cmd)
    + output artifacts (outs)
      + special output data e.g. metrics and plots
      + if a following stage use the output of a stage as input/dependency, DVC automatically connect them in DAG
* using `dvc stage add` to generate dvc pipeline step
  + create stages using `dvc stage add`
  + ```bash
    dvc stage add \
    -n preprocess \ # stage name
    -p params.yaml:preprocess \ # parameters. use params from params.yaml file's preprocess key
    -d raw_data.csv \ # dependency as the input
    -d preprocess.py \ # dependency as execution script file
    -o processed_data.csv \ # outputs as outs
    python3 preprocess.py # this will be the cmd at the end
    ```
  + the output yaml file is the following:
  + ```YAML
    stages:
      preprocess:
        cmd: python3 preprocess.py
        params:
          # keys from params.yaml
          - params.yaml
            - preprocess
        deps:
        - preprocess.py
        - raw_data.csv
        outs:
        - processed_data.csv
    ```
* you can update a stage by `dvc stage add --force` to overwrite the previous stages
  + the following code will update the preprocess stage when using --force
  ```bash
   dvc stage add --force \
   -n preprocess
   ```
* generate dvc.yaml by shell script
  + run the following bash file by `bash dvc_dag_stages_add.sh`
    ```bash
    # example shell script to generate dvc.yaml for dvc pipeline  
    # dvc_dag_stages_add.sh
    #!/bin/bash
    
    # Preprocess stage
    # Output is processed_data.csv
    dvc stage add --force \
    -n preprocess_stage \
    -p preprocess \
    -d raw_data.csv \
    -o processed_data.csv \
    python3 preprocess.py
    
    # split stage
    # this stage uses parameters from 'split' section of params.yaml
    dvc stage add --force \
    -n split_stage \
    -p split \
    -d processed_data.csv \
    -d split.py \
    -o train_data.csv \
    -o eval_data.csv \
    python3 split.py
    
    # train stage
    # This stage generates model.pkl as output
    dvc stage add --force \
    -n train_stage \
    -p train \
    -d train_data.csv \
    -d train.py \
    -o model.pkl \
    python3 train.py
    
    # Evaluate stage
    # This stage uses model.pkl as one of the input
    dvc stage add --force \
    -n evaluate_stage \
    -p evaluate \
    -d eval_data.csv \
    -d model.pkl \
    -d evaluate.py \
    -o metrics.json \
    python3 evaluate.py
    ```
  
* visualize DVC pipelines using `dvc dag`
  + automatically visualize DAG from dvc.yaml file in the project folder
  ```bash
  dvc dag # visualize the entire pipeline dag
  dvc dag <target> # visualize all stages up to a specific target stage
  dvc dag --outs # only display stage outputs as nodes in DAG. or the outputs DAG w/o showing the entire pipeline
  dvc dag --dot # generate DOT script that can be used to generate visualizations useful for documentation
  ``` 
    

#### Executing DVC pipelines
* when you change a dataset that at the begining of the pipeline
  + you first use `dvc add data.csv`, `git add data.csv.dvc`, and `git commit`, then `dvc repro`
  + the following changed data or objects will be added by pipeline about their .dvc files, dvc.yaml and dvc.lock
  + run `git add` to update all .dvc, dvc.yaml and dvc.lock, `git commit` and `git push`
  + run `dvc push` to update the dataset to remote
* what happens when running `dvc repro`
  + A state file dvc.lock is generated
    + this file is similar to .dvc file, captures MD5 hashes
    + .dvc stores individual data file state
    + dvc.lock store the MD5 hashes for each stage executed by `dvc repro <target>` or `dvc repro`
    + It is a good practice to run `git add dvc.lock && git commit -m "first pipeline run"` immediately after we create or modify this file to record the current state and results
  + stage caching in DVC when running `dvc repro`
    + during the stage run, DVC calculates a hash using the md5 checksum for each dependency such as input file, code and parameters.
    + if any dependency has changed since the last run, the hash will be different, and DVC will consider the stage outdated, requiring a rerun
    + DVC also checks if the outputs of a stoage are already cached using the md5 checksum.
      + If they are, and the dependencies and code have not changed, DVC will use the cached outputs without rerunning the stage.
      + otherwise, it will rerun the stage. after execution, DVC calculates a new hash for the outputs and stores the results in the cache for future use
* use `dvc repro --dry` to only print commands without running the pipeline
  + this is useful to verify which stage(s) will be executed
* if we re-run our pipeline but some steps have not changed, DVC will use the previously cached results and skip the execution of those steps. 
* Additional arguments
  + running specific files dvc repro linear/dvc.yaml
    + multiple dvc.yaml in one folder are not allowed
  + running specific stages by `dvc repro <target_stage>`
    + this also run upstream dependencies that the specified stage depends on
  + force run every stage of a pipeline even if they have been executed before by `dvc repro -f`
    + for complete and refresh execution of the pipeline, regardless of any cached results
  + Not storing execution outputs in cache `dvc repro --no-commit` if we don't want to cache the outputs
    + be sure to use `dvc commit` to restore the dvc cache function when we are done
* parallel stage execution
  + we can run independent steps concurrently by launching `dvc repro` multiple times concurrently, for example, in separate terminals


### Evaluation: Metrics and plots in DVC
* using DVC to monitor and evaluate model performance with metrics and plots
* configure DVC YAML file to track metrics across experiments
* track metrics
  + change metrics.json from outs to metrics
  + set cache: false to track it by git rather than dvc, since metric files are small and text-based
  + we use dvc to track large data file and binary, non-text files
```YAML
stages:
  train_and_evaluate:
    outs:
    - plots.png
    metrics:
      - metrics.json
        cache: false
```
  + we can use `dvc metrics show` to print current metrics on terminal
  + we can compare metrics across runs by `dvc metrics diff`
  + we can also track data files used to generate plots by defining "plots" section with <data_file_for_plot>, template that defines the style of plot, column names for x, y, x_label, y_label, title and set cache to false
    + print dvc plot by `dvc plots show <data_file_for_plot>`         